In [1]:
!mkdir -p scraping data

In [2]:
!pip -q install google-play-scraper pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00


In [3]:
%%writefile scraping/scrape_playstore.py
import argparse
import time
import random
import pandas as pd
from tqdm import tqdm
from google_play_scraper import reviews, Sort

def rating_to_label(score: int) -> str:
    if score <= 2:
        return "negatif"
    elif score == 3:
        return "netral"
    else:
        return "positif"

def scrape_app(app_id: str, lang: str, country: str, target_per_app: int, sleep_min: float, sleep_max: float):
    all_rows = []
    continuation_token = None
    pbar = tqdm(total=target_per_app, desc=f"Scraping {app_id}", unit="rev")

    while len(all_rows) < target_per_app:
        count = min(200, target_per_app - len(all_rows))
        result, continuation_token = reviews(
            app_id,
            lang=lang,
            country=country,
            sort=Sort.NEWEST,
            count=count,
            continuation_token=continuation_token
        )

        if not result:
            break

        for r in result:
            all_rows.append({
                "app_id": app_id,
                "reviewId": r.get("reviewId"),
                "userName": r.get("userName"),
                "score": r.get("score"),
                "at": r.get("at"),
                "content": r.get("content"),
                "thumbsUpCount": r.get("thumbsUpCount"),
                "replyContent": r.get("replyContent"),
                "repliedAt": r.get("repliedAt"),
                "label": rating_to_label(int(r.get("score", 0) or 0))
            })

        pbar.update(len(result))

        if continuation_token is None:
            break

        time.sleep(random.uniform(sleep_min, sleep_max))

    pbar.close()
    return all_rows

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--apps", nargs="+", required=True)
    parser.add_argument("--out", default="dataset_raw.csv")
    parser.add_argument("--lang", default="id")
    parser.add_argument("--country", default="id")
    parser.add_argument("--min_total", type=int, default=3000)
    parser.add_argument("--per_app", type=int, default=1400)
    parser.add_argument("--sleep_min", type=float, default=0.5)
    parser.add_argument("--sleep_max", type=float, default=1.2)
    args = parser.parse_args()

    rows = []
    for app in args.apps:
        rows.extend(scrape_app(app, args.lang, args.country, args.per_app, args.sleep_min, args.sleep_max))
        if len(rows) >= args.min_total:
            break

    df = pd.DataFrame(rows)
    if "reviewId" in df.columns:
        df = df.drop_duplicates(subset=["reviewId"])
    df["content"] = df["content"].fillna("").astype(str)
    df = df[df["content"].str.strip().str.len() > 0]
    df.to_csv(args.out, index=False, encoding="utf-8-sig")

    print("\n=== DONE ===")
    print(f"Saved: {args.out}")
    print(f"Total rows: {len(df)}")
    print("Label distribution:")
    print(df["label"].value_counts(dropna=False))

if __name__ == "__main__":
    main()


Writing scraping/scrape_playstore.py


In [4]:
!python scraping/scrape_playstore.py \
  --apps com.shopee.id com.tokopedia.tkpd com.lazada.android \
  --out data/dataset_raw.csv \
  --min_total 3000 \
  --per_app 1600

Scraping com.shopee.id: 100% 1600/1600 [00:08<00:00, 186.88rev/s]
Scraping com.tokopedia.tkpd: 100% 1600/1600 [00:08<00:00, 192.52rev/s]

=== DONE ===
Saved: data/dataset_raw.csv
Total rows: 3200
Label distribution:
label
positif    1996
negatif    1033
netral      171
Name: count, dtype: int64


In [5]:
import pandas as pd
df_raw = pd.read_csv("data/dataset_raw.csv")
df_raw.shape, df_raw["label"].value_counts()

((3200, 10),
 label
 positif    1996
 negatif    1033
 netral      171
 Name: count, dtype: int64)

In [6]:
!zip -r data.zip data

  adding: data/ (stored 0%)
  adding: data/dataset_raw.csv (deflated 71%)


In [7]:
from google.colab import files
files.download("data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>